<a href="https://colab.research.google.com/github/ashu433/Machine-Learning-Book-Practice-Q-A/blob/main/ML_Traiding_Startjee_Development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd
import numpy as np
from google.colab import drive
import os
from sklearn.preprocessing import MinMaxScaler
import glob
import re
import gc
from tensorflow import keras
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam
import json
from tensorflow.keras.callbacks import TensorBoard
import datetime
import shutil
import tensorflow as tf
import sys

In [19]:
# import sys

# # Reset stdout and stderr to the default streams (console)
# sys.stdout = sys.__stdout__
# sys.stderr = sys.__stderr__

In [20]:
print("Ashu")

Ashu


In [21]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Constants**

In [22]:
Time=5
Path_training_data=f"/content/drive/MyDrive/Training Data/Training Data {Time} min/"
checkpoint_dir = '/content/drive/MyDrive/Saved training parameters/'
Model_parameters_path="/content/drive/MyDrive/model_performance_parameter/"
Path_residual_files="/content/drive/MyDrive/Residual Files/"

# **Data Prepration**

In [23]:
def preprocessing(df,one_hot_cols):
    exclude_cols = ['Date', 'Time']
    feature_cols = [col for col in required_columns if col not in exclude_cols + one_hot_cols]

    scaler = MinMaxScaler()
    scaled_features = scaler.fit_transform(df[feature_cols])
    scaled_df = pd.DataFrame(scaled_features, columns=feature_cols)

    # Concatenate Date, Time, unscaled one-hot columns, and scaled features
    final_df = pd.concat([
        df[exclude_cols].reset_index(drop=True),
        scaled_df.reset_index(drop=True),
        df[one_hot_cols].reset_index(drop=True)
    ], axis=1)

    # Reorder to match original layout
    final_df = final_df[[col for col in required_columns if col in final_df.columns]]

    return final_df


In [24]:
def replace_and_scale_strike_features(df):
    # List of original strike columns
    strike_columns = [
        "ATM Strike",
        "ATM Strike plus 1",
        "ATM Strike plus 2",
        "ATM Strike minus 1",
        "ATM Strike minus 2"
    ]

    # Get the column positions of each original strike column
    strike_positions = [df.columns.get_loc(col) for col in strike_columns]

    # Compute new moneyness-style features
    df["Relative_ATM_Strike"]   = (df["ATM Strike"] - df["Index Price"]) / df["Index Price"]
    df["Moneyness_CE_plus1"]    = (df["ATM Strike plus 1"] - df["Index Price"]) / df["Index Price"]
    df["Moneyness_CE_plus2"]    = (df["ATM Strike plus 2"] - df["Index Price"]) / df["Index Price"]
    df["Moneyness_PE_minus1"]   = (df["Index Price"] - df["ATM Strike minus 1"]) / df["Index Price"]
    df["Moneyness_PE_minus2"]   = (df["Index Price"] - df["ATM Strike minus 2"]) / df["Index Price"]

    # Store new columns
    new_features = [
        "Relative_ATM_Strike",
        "Moneyness_CE_plus1",
        "Moneyness_CE_plus2",
        "Moneyness_PE_minus1",
        "Moneyness_PE_minus2"
    ]

    # Drop original strike columns
    df.drop(columns=strike_columns, inplace=True)

    # Insert new columns at the original positions
    for new_col, pos in zip(new_features, strike_positions):
        col_series = df.pop(new_col)
        df.insert(loc=pos, column=new_col, value=col_series)

    # Apply MinMax scaling to the new columns

    return df

In [25]:
def update_required_columns(required_columns):
    # Columns to remove and corresponding columns to insert
    replacements = {
        "ATM Strike": "Relative_ATM_Strike",
        "ATM Strike plus 1": "Moneyness_CE_plus1",
        "ATM Strike plus 2": "Moneyness_CE_plus2",
        "ATM Strike minus 1": "Moneyness_PE_minus1",
        "ATM Strike minus 2": "Moneyness_PE_minus2",
    }

    # Make a copy so original list is not modified outside
    updated_columns = required_columns.copy()

    for old_col, new_col in replacements.items():
        if old_col in updated_columns:
            idx = updated_columns.index(old_col)
            updated_columns.pop(idx)
            updated_columns.insert(idx, new_col)

    return updated_columns

In [26]:
def build_multi_output_model(input_shape, lstm_units=20, num_lstm_layers=3, learning_rate=0.001):
    inputs = Input(shape=input_shape)

    x = inputs
    for _ in range(num_lstm_layers - 1):
        x = layers.LSTM(lstm_units, return_sequences=True)(x)
    x = layers.LSTM(lstm_units)(x)

    # Output 1: Sentiment (2 classes, softmax)
    sentiment_output = layers.Dense(2, activation='softmax', name='sentiment')

    # Output 2: Relative Body % (binary classification)
    body_output = layers.Dense(1, activation='sigmoid', name='body')

    model = Model(inputs=inputs, outputs=[sentiment_output(x), body_output(x)])

    model.compile(
        loss={
            'sentiment': 'categorical_crossentropy',
            'body': 'binary_crossentropy'
        },
        optimizer=Adam(learning_rate=learning_rate),
        metrics={
            'sentiment': 'accuracy',
            'body': 'accuracy'
        }
    )
    return model

In [27]:
def reading_date_list(text_file="All_Dates_list.txt"):
    with open(Path_residual_files + text_file, 'r') as file:
        json_data = file.read()
        present_market_status = json.loads(json_data)
        return present_market_status

In [28]:
def writing_market_status(dict_name, text_file="All_Dates_list.txt"):
    with open(Path_residual_files + text_file, 'w') as file:
        json.dump(dict_name, file)

In [29]:
target_columns = [
    'Sentiment of day_Bearish',
    'Sentiment of day_Bullish',
    'Relative Body of candle %'
]


In [30]:
# import sys
# import os

# class Tee:
#     def __init__(self, *streams):
#         self.streams = streams

#     def write(self, data):
#         for s in self.streams:
#             s.write(data)
#             s.flush()

#     def flush(self):
#         for s in self.streams:
#             s.flush()

# # Ensure you define Path_residual_files before using it.
#  # or provide your directory path

# log_file_path = os.path.join(Path_residual_files, "training_log.txt")
# log_file = open(log_file_path, "w")
# sys.stdout = sys.stderr = Tee(sys.__stdout__, log_file)

# print("Ashutosh")  # This should print to both console and log file


In [31]:
# log_file_path = os.path.join(Path_residual_files, "training_log.txt")
# log_file = open(log_file_path, "w")
# sys.stdout = sys.stderr = Tee(sys.__stdout__, log_file)

In [32]:
# print("Ashutosh")

# **Training Of Model**

In [33]:
filtered_df=pd.read_csv(Path_residual_files+"Output_file.csv")

In [34]:
filtered_df['Date'] = pd.to_datetime(filtered_df['Date'], format='%d-%m-%Y').dt.strftime('%d-%m-%Y')

# **Building Model**

In [35]:
Date_log=reading_date_list()
Date_train_final=Date_log["Remining Training Dates"]
Date_validation=Date_log["Validation Dates"]
Dates_already_trained=Date_log["Completed Training Dates"]

Training_date_check=[Date_train_final[0],Date_train_final[1],Date_train_final[2],Date_train_final[3]]
Validation_date_check=[Date_validation[0],Date_validation[1],Date_validation[2],Date_validation[3]]

print(f"Training Date Check: {Training_date_check}")
print(f"Validation Date Check: {Validation_date_check}")

Training Date Check: ['16-12-2021', '19-08-2024', '08-02-2024', '13-09-2023']
Validation Date Check: ['28-04-2021', '31-01-2022', '26-10-2021', '10-07-2020']


In [36]:
import tensorflow as tf
tf.config.run_functions_eagerly(True)

In [37]:
# Define log file path


# === Prepare TensorBoard log directory ===
tensorboard_base_log_dir = os.path.join(Model_parameters_path, "logs")
shutil.rmtree(tensorboard_base_log_dir, ignore_errors=True)
validation_writer = tf.summary.create_file_writer(os.path.join(tensorboard_base_log_dir, "validation"))

Initiation=1
Resume=0
train_counter = 0
Validation_after_training_days=2


if Initiation==1:
  Date_log=reading_date_list()
  Date_train_final=Date_log["Remining Training Dates"]
  Date_validation=Date_log["Validation Dates"]
  Dates_already_trained=Date_log["Completed Training Dates"]

  model=load_model(checkpoint_dir+"initial_model.h5")
  model.compile(
    loss={
        'sentiment': 'categorical_crossentropy',
        'body': 'binary_crossentropy'
    },
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics={
        'sentiment': 'accuracy',
        'body': 'accuracy'
    }
    )
elif Resume==1:
  Date_log=reading_date_list()
  Date_train_final=Date_log["Remining Training Dates"]
  Date_validation=Date_log["Validation Dates"]
  Dates_already_trained=Date_log["Completed Training Dates"]

  Target_date=Date_train_final[0]
  model_name=f"model_after_{Target_date}.h5"
  model=load_model(checkpoint_dir+model_name)

  model.compile(
    loss={
        'sentiment': 'categorical_crossentropy',
        'body': 'binary_crossentropy'
    },
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics={
        'sentiment': 'accuracy',
        'body': 'accuracy'
    }
    )
else:
  pass

training_history_path = os.path.join(Model_parameters_path, 'training_history.csv')
validation_history_path = os.path.join(Model_parameters_path, 'validation_history.csv')

for date_str in Training_date_check:
  try:
      Date_log=reading_date_list()
      required_columns=Date_log["required_columns"]
      one_hot_cols=Date_log["one_hot_cols"]

      print(f"Running the Training for the Date: {date_str}")
      formatted_date = pd.to_datetime(date_str, format='%d-%m-%Y').strftime('%Y-%m-%d')
      file_name = f"Training_Data_{formatted_date}.csv"
      file_path = os.path.join(Path_training_data, file_name)

      if os.path.exists(file_path):
          df = pd.read_csv(file_path)

          df = df[[col for col in df.columns if col in required_columns]]
          df = df[[col for col in required_columns if col in df.columns]]

          df = replace_and_scale_strike_features(df)
          required_columns = update_required_columns(required_columns)
          df = preprocessing(df, one_hot_cols)

          df = df.drop(columns=['Date', 'Time'], errors='ignore')

          row = filtered_df[filtered_df['Date'] == date_str]
          if row.empty:
              print(f"No output labels found for {date_str}. Skipping.")
              continue

          y_target = row[target_columns].values.reshape(1, len(target_columns))
          X = df.values.reshape(1, -1, df.shape[1])

          y_sentiment = y_target[:, :2]  # First 2 values are for 'sentiment' softmax
          y_body = y_target[:, 2:]       # Last value is for 'body' sigmoid

          # === TensorBoard log for this training date ===
          log_dir = os.path.join(tensorboard_base_log_dir, f"{date_str}_{datetime.datetime.now().strftime('%H%M%S')}")
          tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

          print(f"X shape: {X.shape}, y_sentiment shape: {y_sentiment.shape}, y_body shape: {y_body.shape}")
          if np.isnan(X).any() or np.isinf(X).any():
              print(f"❌ Found NaNs/Infs in X for {date_str}. Skipping.")
          if np.isnan(y_sentiment).any() or np.isnan(y_body).any():
              print(f"❌ Found NaNs/Infs in target for {date_str}. Skipping.")

          # === Train model ===
          history = model.fit(
              X,
              {'sentiment': y_sentiment, 'body': y_body},
              epochs=50,
              verbose=1
              # callbacks=[tensorboard_callback]
          )

          # Console print
          final_train_loss = history.history['loss'][-1]
          print(f"✅ Final Training Loss for {date_str}: {final_train_loss:.4f}")
          for key in history.history:
              if key != 'loss':
                  print(f"{key} for {date_str}: {history.history[key][-1]:.4f}")

          # Save training history
          history_df = pd.DataFrame(history.history)
          history_df['Date'] = date_str
          history_df.to_csv(training_history_path, mode='a', header=not os.path.exists(training_history_path), index=False)

          # === Save checkpoint after training ===
          model_checkpoint_path = os.path.join(checkpoint_dir, f"model_after_{date_str}.h5")
          model.save(model_checkpoint_path)
          print(f"Model checkpoint saved after training on {date_str}.")

          Date_log["Remining Training Dates"].remove(date_str)
          Date_log["Completed Training Dates"].append(date_str)
          writing_market_status(Date_log, "All_Dates_list.txt")
          # === CLEAR memory ===
          del df, X, y_target, history, history_df
          gc.collect()

          # === Increment training counter ===
          train_counter += 1

          # === Perform validation after every 2 trained days ===
          if train_counter % Validation_after_training_days == 0:
              print("\n--- Running validation on validation set ---\n")
              for val_date in Validation_date_check:
                    try:
                        Date_log=reading_date_list()
                        required_columns=Date_log["required_columns"]
                        one_hot_cols=Date_log["one_hot_cols"]

                        print(f"Running the Validation for the Date: {val_date}")
                        val_formatted = pd.to_datetime(val_date, format='%d-%m-%Y').strftime('%Y-%m-%d')
                        val_file = f"Training_Data_{val_formatted}.csv"
                        val_path = os.path.join(Path_training_data, val_file)

                        if not os.path.exists(val_path):
                            print(f"Validation file not found: {val_date}")
                            continue

                        val_df = pd.read_csv(val_path)
                        val_df = val_df[[col for col in val_df.columns if col in required_columns]]
                        val_df = val_df[[col for col in required_columns if col in val_df.columns]]

                        val_df = replace_and_scale_strike_features(val_df)
                        required_columns = update_required_columns(required_columns)
                        val_df = preprocessing(val_df, one_hot_cols)

                        val_df = val_df.drop(columns=['Date', 'Time'], errors='ignore')

                        val_row = filtered_df[filtered_df['Date'] == val_date]
                        if val_row.empty:
                            print(f"No output labels found for validation date {val_date}. Skipping.")
                            continue

                        y_val = val_row[target_columns].values.reshape(1, len(target_columns))
                        X_val = val_df.values.reshape(1, -1, val_df.shape[1])


                        y_val_sentiment = y_val[:, :2]
                        y_val_body = y_val[:, 2:]

                        # === Evaluate ===
                        val_result = model.evaluate(X_val,
                                                    {'sentiment': y_val_sentiment, 'body': y_val_body},
                                                    verbose=1)
                        metrics_names = model.metrics_names

                        print(f"📊 Validation Results for {val_date}:")
                        for name, value in zip(metrics_names, val_result):
                            print(f"  {name}: {value:.4f}")

                        val_record = dict(zip(metrics_names, val_result))
                        val_record["Date"] = val_date

                        val_df_save = pd.DataFrame([val_record])
                        val_df_save.to_csv(validation_history_path, mode='a', header=not os.path.exists(validation_history_path), index=False)

                        # Log to TensorBoard
                        with validation_writer.as_default():
                            for name, value in zip(metrics_names, val_result):
                                tf.summary.scalar(f"val_{name}", value, step=train_counter)
                        validation_writer.flush()

                        del val_df, X_val, y_val, val_df_save
                        gc.collect()

                    except Exception as ve:
                        print(f"Error validating on {val_date}: {ve}")

  except Exception as e:
      print(f"Error processing date {date_str}: {e}")

Running the Training for the Date: 16-12-2021
X shape: (1, 75, 63), y_sentiment shape: (1, 2), y_body shape: (1, 1)
Epoch 1/50


/usr/local/lib/python3.11/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - body_accuracy: 1.0000 - body_loss: 0.6817 - loss: 1.2721 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.5904
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - body_accuracy: 1.0000 - body_loss: 0.6616 - loss: 1.2041 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.5425
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - body_accuracy: 1.0000 - body_loss: 0.6414 - loss: 1.1388 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.4974
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - body_accuracy: 1.0000 - body_loss: 0.6208 - loss: 1.0767 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.4559
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - body_accuracy: 1.0000 - body_loss: 0.5984 - loss: 1.0177 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.4193
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - body_accuracy: 1.0000 - body_loss: 0.5730 - loss: 0.9609 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.3879
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - bo

✅ Final Training Loss for 16-12-2021: 0.0887
body_accuracy for 16-12-2021: 1.0000
body_loss for 16-12-2021: 0.0503
sentiment_accuracy for 16-12-2021: 1.0000
sentiment_loss for 16-12-2021: 0.0384
Model checkpoint saved after training on 16-12-2021.
Running the Training for the Date: 19-08-2024
X shape: (1, 75, 63), y_sentiment shape: (1, 2), y_body shape: (1, 1)
Epoch 1/50


/usr/local/lib/python3.11/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 861ms/step - body_accuracy: 1.0000 - body_loss: 0.4119 - loss: 0.5700 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.1581
📊 Validation Results for 31-01-2022:
  loss: 0.5700
  compile_metrics: 0.1581
  sentiment_loss: 0.4119
  body_loss: 1.0000
Running the Validation for the Date: 26-10-2021


/usr/local/lib/python3.11/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 834ms/step - body_accuracy: 0.0000e+00 - body_loss: 1.0840 - loss: 1.2415 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.1575
📊 Validation Results for 26-10-2021:
  loss: 1.2415
  compile_metrics: 0.1575
  sentiment_loss: 1.0840
  body_loss: 0.0000
Running the Validation for the Date: 10-07-2020


/usr/local/lib/python3.11/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 856ms/step - body_accuracy: 1.0000 - body_loss: 0.4114 - loss: 0.5698 - sentiment_accuracy: 1.0000 - sentiment_loss: 0.1584
📊 Validation Results for 10-07-2020:
  loss: 0.5698
  compile_metrics: 0.1584
  sentiment_loss: 0.4114
  body_loss: 1.0000
